## Convolution Autoencoder

In [ ]:
import numpy as np                          # Import numpy for numerical operations
from tensorflow.keras.datasets import mnist # Import MNIST dataset (digits 0–9)
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D # Layers used in CNN autoencoder
from tensorflow.keras.models import Model   # Used to define the model
import matplotlib.pyplot as plt             # For visualization

# Load data
(x_train, _), (x_test, _) = mnist.load_data()  
# Loads MNIST dataset
# x_train, x_test = images
# _ means labels are ignored (autoencoder doesn't need labels)

# Normalize and reshape
x_train = x_train.astype('float32') / 255.  
# Convert pixel values (0–255) to float and normalize to range [0,1]

x_test = x_test.astype('float32') / 255.    
# Same normalization for test data

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))  
# Reshape training data to (num_samples, height, width, channels)
# MNIST is grayscale → 1 channel

x_test = np.reshape(x_test, (len(x_test), 28, 28, 1))    
# Same reshape for test data

# Encoder
input_img = Input(shape=(28,28,1))  
# Define input layer with image size 28x28x1

x = Conv2D(16, (3,3), activation='relu', padding='same')(input_img)  
# Apply convolution:
# 16 filters, kernel size 3x3
# ReLU activation
# padding='same' keeps output size same (28x28)

x = MaxPooling2D((2,2), padding='same')(x)   
# Downsample image by factor 2 → (28x28 → 14x14)
# Keeps important features

x = Conv2D(32, (3,3), activation='relu', padding='same')(x)  
# Another convolution with 32 filters for deeper feature extraction

encoded = MaxPooling2D((2,2), padding='same')(x)  
# Further downsampling → (14x14 → 7x7)
# This is the compressed representation (latent space)

# Decoder
x = Conv2D(32, (3,3), activation='relu', padding='same')(encoded)  
# Start decoding: extract features again from compressed data

x = UpSampling2D((2,2))(x)   
# Upsample → (7x7 → 14x14)
# Reverse of pooling

x = Conv2D(16, (3,3), activation='relu', padding='same')(x)  
# Reduce channels back while reconstructing

x = UpSampling2D((2,2))(x)   
# Upsample again → (14x14 → 28x28)

decoded = Conv2D(1, (3,3), activation='sigmoid', padding='same')(x)  
# Final output layer:
# 1 channel (grayscale)
# sigmoid → output in range [0,1] (same as input)

# Model
autoencoder = Model(input_img, decoded)  
# Define model: input → reconstructed output

autoencoder.compile(optimizer='adam', loss='mse')  
# Compile model:
# optimizer='adam' → adjusts weights
# loss='mse' → measures reconstruction error

# Train
autoencoder.fit(x_train, x_train,
                epochs=5,
                batch_size=128,
                shuffle=True,
                validation_data=(x_test, x_test))
# Train model:
# input = x_train, output = x_train (autoencoder learns to reconstruct)
# epochs=5 → number of full passes over dataset
# batch_size=128 → process 128 images at a time
# shuffle=True → randomize data each epoch
# validation_data → evaluate on test data

# Reconstruction
decoded_imgs = autoencoder.predict(x_test)  
# Generate reconstructed images from test set

# Visualization
n = 5  
# Number of images to display

for i in range(n):
    plt.subplot(2,n,i+1)
    plt.imshow(x_test[i].reshape(28,28), cmap='gray')
    plt.axis('off')
    # Show original image

    plt.subplot(2,n,i+n+1)
    plt.imshow(decoded_imgs[i].reshape(28,28), cmap='gray')
    plt.axis('off')
    # Show reconstructed image

plt.show()  
# Display all images

## Denoising Autoencoders


import numpy as np  
# Used for numerical operations and generating random noise

from tensorflow.keras.datasets import mnist  
# Loads MNIST dataset (28x28 grayscale digit images)

from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D  
# Layers used to build convolutional autoencoder:
# Conv2D → feature extraction
# MaxPooling → downsampling (encoder)
# UpSampling → upscaling (decoder)

from tensorflow.keras.models import Model  
# Used to define full model (input → output)

# Load data
(x_train, _), (x_test, _) = mnist.load_data()  
# x_train, x_test → images
# _ → labels ignored (autoencoder is unsupervised)

# Normalize
x_train = x_train.astype('float32') / 255.  
# Convert pixel values from [0,255] → [0,1]

x_test = x_test.astype('float32') / 255.  

# Reshape
x_train = x_train.reshape((-1,28,28,1))  
# Convert to (samples, height, width, channels)
# 1 channel → grayscale

x_test = x_test.reshape((-1,28,28,1))  

# Add noise
noise_factor = 0.5  
# Controls amount of noise (higher = more noise)

x_train_noisy = x_train + noise_factor * np.random.normal(size=x_train.shape)  
# Add Gaussian noise to training images

x_test_noisy = x_test + noise_factor * np.random.normal(size=x_test.shape)  

# Clip values to [0,1]
x_train_noisy = np.clip(x_train_noisy,0,1)  
# Ensures pixel values remain valid after noise addition

x_test_noisy = np.clip(x_test_noisy,0,1)  

# ================= ENCODER =================
input_img = Input(shape=(28,28,1))  
# Input layer: image size

x = Conv2D(32,(3,3),activation='relu',padding='same')(input_img)  
# 32 filters → detect edges, patterns
# kernel = 3x3
# ReLU → non-linearity
# padding='same' → output size remains 28x28

x = MaxPooling2D((2,2),padding='same')(x)  
# Downsample → (28x28 → 14x14)
# Reduces spatial size → compression

x = Conv2D(64,(3,3),activation='relu',padding='same')(x)  
# Increase filters → learn more complex features

encoded = MaxPooling2D((2,2),padding='same')(x)  
# Further compression → (14x14 → 7x7)
# This is latent representation (compressed form)

# ================= DECODER =================
x = Conv2D(64,(3,3),activation='relu',padding='same')(encoded)  
# Start reconstruction from compressed features

x = UpSampling2D((2,2))(x)  
# Upsample → (7x7 → 14x14)

x = Conv2D(32,(3,3),activation='relu',padding='same')(x)  
# Reduce feature depth

x = UpSampling2D((2,2))(x)  
# Upsample → (14x14 → 28x28)

decoded = Conv2D(1,(3,3),activation='sigmoid',padding='same')(x)  
# Final output:
# 1 channel image
# sigmoid → ensures output in [0,1]

# ================= MODEL =================
model = Model(input_img, decoded)  
# Defines mapping: noisy image → clean image

model.compile(optimizer='adam', loss='mse')  
# optimizer='adam' → adaptive learning (fast convergence)
# loss='mse' → reconstruction error (||x - y||²)

# ================= TRAIN =================
model.fit(x_train_noisy, x_train,
          epochs=5,
          batch_size=128,
          validation_data=(x_test_noisy,x_test))  
# Input = noisy image
# Output = clean image
# Model learns to remove noise
